## **XG Boost** ##

In [11]:
import pandas as pd 
import numpy as np

from scipy.stats import randint, uniform
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score

In [3]:
df = pd.read_csv("data.csv")
df["From"] = pd.to_datetime(df["From"])
df = df.sort_values("From").reset_index(drop=True)

In [4]:
mete_col = ["Diurnal temperature range (°C)", "Mean temperature (°C)", "Mean humidity (%)", "Solar radiation (MJ/m²)", "Air pressure (hPa)", "Dew point temperature (°C)", "Wind speed (km/h)", "Mean UV"]
# flu_col = ["H1", "H3", "B", "AandB"]
flu_col = ["H1", "H3", "AandB"]

In [20]:
for col in flu_col: 
    tmp_df = df.copy()

    # Seperate case for type AH1 influenza
    if col == "H1":
        for n in range(4):
            tmp_df[f"{col}_Lag{n+1}"] = tmp_df[col].shift(n+1)
        tmp_df[f"{col}_volatility"] = tmp_df[col].shift(1).rolling(window=4).std()
        tmp_df[f"{col}_rolling_mean"] = tmp_df[col].shift(1).rolling(window=4).mean()
    else:
        for n in range(2):
            tmp_df[f"{col}_Lag{n+1}"] = tmp_df[col].shift(n+1)
        tmp_df[f"{col}_volatility"] = tmp_df[col].shift(1).rolling(window=4).std()
        tmp_df[f"{col}_rolling_mean"] = tmp_df[col].shift(1).rolling(window=4).mean()

    tmp_df = tmp_df.dropna().reset_index(drop=True)

    X = tmp_df.drop(columns=["From"]+flu_col)


    print(f"{'='*70} {col} {'='*70}")
    y = tmp_df[col]

    train_end = int(tmp_df.shape[0] * 0.8)
    X_train, X_test = X.iloc[:train_end], X.iloc[train_end:]
    y_train, y_test = y.iloc[:train_end], y.iloc[train_end:]

    y_train_log = np.log1p(y_train)

    grid = {
        "n_estimators": [x for x in range(100, 1000, 100)],
        "learning_rate": uniform(0.01, 0.20),
        "max_depth": [3, 4, 5, 6, 7, 8],
        "subsample": uniform(0.6, 0.4),
        "colsample_bytree": uniform(0.6, 0.4),
        "min_child_weight": randint(10, 30),
        "reg_alpha": uniform(0, 5),
        "reg_lambda": uniform(1, 10)
    }

    tscv = TimeSeriesSplit(n_splits=5)
    xgb = XGBRegressor(random_state=42)
    xgb_cv = RandomizedSearchCV(estimator=xgb, param_distributions=grid, cv=tscv, scoring="roc_auc", n_jobs=-1)

    xgb_cv.fit(X_train, y_train_log)

    print(f"Best Parameters: {xgb_cv.best_params_}")

    print(f"\nFeature importances: ")
    model = xgb_cv.best_estimator_
    importance = pd.Series(model.feature_importances_, index=X.columns)
    print(importance.sort_values(ascending=False))

    model.fit(X_train, y_train_log)
    
    # Evaluation
    y_pred_log = model.predict(X_train)
    y_pred = np.expm1(y_pred_log)
    print(f"\nModel metrics with train data: ")
    print(f"MAE: {mean_absolute_error(y_train, y_pred):.4f}")
    print(f"MSE: {mean_squared_error(y_train, y_pred):.4f}")
    print(f"R2:  {r2_score(y_train, y_pred):.4f}")

    y_pred_log = model.predict(X_test)
    y_pred = np.expm1(y_pred_log)
    print(f"\nModel metrics with test data: ")
    print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")
    print(f"MSE: {mean_squared_error(y_test, y_pred):.4f}")
    print(f"R2:  {r2_score(y_test, y_pred):.4f}")



====================================================================== H1 ======================================================================


c:\Users\yanss\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:1135: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


Best Parameters: {'colsample_bytree': np.float64(0.9774204252457401), 'learning_rate': np.float64(0.12474503611678893), 'max_depth': 6, 'min_child_weight': 29, 'n_estimators': 600, 'reg_alpha': np.float64(4.819910077212871), 'reg_lambda': np.float64(5.852311525461044), 'subsample': np.float64(0.900735673425898)}

Feature importances: 
H1_Lag1                           0.599430
H1_rolling_mean                   0.292413
H1_Lag2                           0.079125
Week                              0.003241
H1_Lag4                           0.002889
H1_Lag3                           0.002389
Dew point temperature (°C)        0.002140
H1_volatility                     0.002058
Mean temperature (°C)             0.002037
B                                 0.002013
Air pressure (hPa)                0.001500
Population                        0.001491
Wind speed (km/h)                 0.001436
Year                              0.001262
Mean UV                           0.001236
Mean humidity (%) 

c:\Users\yanss\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:1135: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


Best Parameters: {'colsample_bytree': np.float64(0.7807927076025925), 'learning_rate': np.float64(0.1948331316434768), 'max_depth': 4, 'min_child_weight': 13, 'n_estimators': 100, 'reg_alpha': np.float64(4.183725562184858), 'reg_lambda': np.float64(4.345847644310514), 'subsample': np.float64(0.6553606042284996)}

Feature importances: 
H3_Lag1                           0.860090
H3_Lag2                           0.026189
H3_rolling_mean                   0.024965
Dew point temperature (°C)        0.008993
Year                              0.007994
Air pressure (hPa)                0.007400
Week                              0.007149
B                                 0.007144
H3_volatility                     0.006803
Population                        0.006290
Mean temperature (°C)             0.005768
Solar radiation (MJ/m²)           0.005466
Log_population                    0.005415
Mean humidity (%)                 0.004877
Wind speed (km/h)                 0.003928
Mean UV           

c:\Users\yanss\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:1135: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


Best Parameters: {'colsample_bytree': np.float64(0.9684804871449301), 'learning_rate': np.float64(0.023188815998898472), 'max_depth': 6, 'min_child_weight': 10, 'n_estimators': 900, 'reg_alpha': np.float64(0.9525053848206572), 'reg_lambda': np.float64(6.897240218298433), 'subsample': np.float64(0.9361775578033485)}

Feature importances: 
AandB_Lag1                        0.802616
Year                              0.101064
AandB_rolling_mean                0.028594
AandB_Lag2                        0.015145
B                                 0.011749
ContainmentHealthIndex_Average    0.010189
AandB_volatility                  0.003809
Mean temperature (°C)             0.003270
Week                              0.002952
school_flag                       0.002919
Air pressure (hPa)                0.002416
Population                        0.002268
Dew point temperature (°C)        0.002252
Solar radiation (MJ/m²)           0.002227
Mean humidity (%)                 0.002162
Diurnal tempera